# Main Rerun Stage 2 Follow-up


# Main Rerun Stage 2 Targeted Follow-up Runtime Config

- 当前 Stage 1 final variant：`linear_svm__sigmoid`
- compare 范围固定为：
  - models: `ridge`, `random_forest`, `xgboost_regressor`
  - specs: `macro_only`, `two_stage_v2_core`

In [1]:
from pathlib import Path
import os

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "plan_v2.md").exists() and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent

os.chdir(PROJECT_ROOT)

MAIN_RERUN_STAGE2_DIR = PROJECT_ROOT / "main_rerun" / "artifacts" / "stage2"
MAIN_RERUN_STAGE2_TARGETED_DIR = MAIN_RERUN_STAGE2_DIR / "final_compare_targeted_followup"
MAIN_RERUN_STAGE2_TARGETED_DIR.mkdir(parents=True, exist_ok=True)

os.environ["STAGE2_HOLDOUT_LABEL"] = "primary_holdout"
os.environ["STAGE2_HOLDOUT_DISPLAY_TITLE"] = "Primary Holdout"
os.environ["STAGE2_SEARCH_PROFILE"] = "targeted_followup"
os.environ["STAGE2_COMPARE_DATA_PATH"] = str(
    MAIN_RERUN_STAGE2_DIR / "stage2_daily_features_long_v2_main_rerun.csv"
)
os.environ["STAGE2_COMPARE_RESULTS_DIR"] = str(MAIN_RERUN_STAGE2_TARGETED_DIR)
os.environ["STAGE2_ACTIVE_VARIANTS"] = "linear_svm__sigmoid"
os.environ["STAGE2_ACTIVE_MODELS"] = "ridge,random_forest,xgboost_regressor"
os.environ["STAGE2_ACTIVE_MODEL_SPECS"] = "macro_only,two_stage_v2_core"

print("Project root:", PROJECT_ROOT)
print("Main rerun Stage 2 feature dir:", MAIN_RERUN_STAGE2_DIR)
print("Main rerun Stage 2 targeted dir:", MAIN_RERUN_STAGE2_TARGETED_DIR)
print("Stage 1 final variant:", "linear_svm__sigmoid")
print("STAGE2_SEARCH_PROFILE =", os.environ["STAGE2_SEARCH_PROFILE"])

Project root: /Users/horace/Coding/ML finance/project
Main rerun Stage 2 feature dir: /Users/horace/Coding/ML finance/project/main_rerun/artifacts/stage2
Main rerun Stage 2 targeted dir: /Users/horace/Coding/ML finance/project/main_rerun/artifacts/stage2/final_compare_targeted_followup
Stage 1 final variant: linear_svm__sigmoid
STAGE2_SEARCH_PROFILE = targeted_followup


In [2]:
from pathlib import Path
import json
import os

os.environ["MPLCONFIGDIR"] = str(Path.cwd() / ".mpl-cache")

import warnings

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBRegressor

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="talk")

DATA_PATH = Path("stage2/artifacts/stage2_daily_features_long_v2.csv")
SEARCH_PROFILE = os.environ.get("STAGE2_SEARCH_PROFILE", "smoke")
RESULTS_DIR = Path(f"stage2/artifacts/nested_cv_model_compare_{SEARCH_PROFILE}_v1")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

TARGET_COL = "gold_ret_1d"
DATE_COL = "date"
TRAIN_LABEL = "train_pool"
HOLDOUT_LABEL = os.environ.get("STAGE2_HOLDOUT_LABEL", "primary_holdout")
HOLDOUT_DISPLAY_TITLE = os.environ.get("STAGE2_HOLDOUT_DISPLAY_TITLE", "Primary Hold-out")
DATA_PATH = Path(os.environ.get("STAGE2_COMPARE_DATA_PATH", str(DATA_PATH)))
RESULTS_DIR = Path(os.environ.get("STAGE2_COMPARE_RESULTS_DIR", str(RESULTS_DIR)))
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

ACTIVE_VARIANTS = [
    key.strip()
    for key in os.environ.get("STAGE2_ACTIVE_VARIANTS", "").split(",")
    if key.strip()
]

MODEL_CANDIDATES = ["ridge", "random_forest", "xgboost_regressor"]
ACTIVE_MODELS = [
    key.strip()
    for key in os.environ.get("STAGE2_ACTIVE_MODELS", ",".join(MODEL_CANDIDATES)).split(",")
    if key.strip()
]
unknown_models = [key for key in ACTIVE_MODELS if key not in MODEL_CANDIDATES]
if unknown_models:
    raise ValueError(f"Unknown STAGE2_ACTIVE_MODELS entries: {unknown_models}")

OUTER_N_SPLITS = 5
OUTER_PURGE_DAYS = 2
OUTER_EMBARGO_DAYS = 5
OUTER_MIN_TRAIN_DAYS = 180
OUTER_MIN_VALID_DAYS = 30

INNER_N_SPLITS = 3
INNER_PURGE_DAYS = 2
INNER_EMBARGO_DAYS = 2
INNER_MIN_TRAIN_DAYS = 120
INNER_MIN_VALID_DAYS = 20

GAP_SPLIT_DAYS = 30
MIN_PREFIX_DAYS_LATER_SEGMENT = 30
RANDOM_STATE = 42

In [3]:
P_TACO_V2_CORE_FEATURES = [
    "p_taco_any",
    "p_taco_top1",
    "p_taco_tail_010",
    "p_taco_any_decay_2d",
]

P_TACO_V2_INTERACTION_FEATURES = [
    "p_taco_any_x_vix_ma_5d_lag1",
    "p_taco_any_x_spx_drawdown_5d_lag1",
]

CORE_CONTROL_FEATURES = [
    "dxy_ret_1d_lag1",
    "real_yield_change_5d_lag1",
    "vix_ma_5d_lag1",
    "vix_change_5d_lag1",
    "spx_ret_1d_lag1",
    "spx_drawdown_5d_lag1",
    "gold_spx_corr_20d_lag1",
]

MODEL_SPEC_CATALOG = {
    "macro_only": CORE_CONTROL_FEATURES,
    "two_stage_v2_core": P_TACO_V2_CORE_FEATURES + CORE_CONTROL_FEATURES,
    "two_stage_v2_interact": P_TACO_V2_CORE_FEATURES + P_TACO_V2_INTERACTION_FEATURES + CORE_CONTROL_FEATURES,
}
ACTIVE_MODEL_SPEC_NAMES = [
    key.strip()
    for key in os.environ.get(
        "STAGE2_ACTIVE_MODEL_SPECS",
        "macro_only,two_stage_v2_core",
    ).split(",")
    if key.strip()
]
unknown_model_specs = [key for key in ACTIVE_MODEL_SPEC_NAMES if key not in MODEL_SPEC_CATALOG]
if unknown_model_specs:
    raise ValueError(f"Unknown STAGE2_ACTIVE_MODEL_SPECS entries: {unknown_model_specs}")
MODEL_SPECS = {key: MODEL_SPEC_CATALOG[key] for key in ACTIVE_MODEL_SPEC_NAMES}


def compute_regression_metrics(y_true, y_pred, benchmark_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    benchmark_pred = np.asarray(benchmark_pred, dtype=float)
    benchmark_mse = mean_squared_error(y_true, benchmark_pred)
    if benchmark_mse == 0:
        oos_r2 = np.nan
    else:
        oos_r2 = 1.0 - (mean_squared_error(y_true, y_pred) / benchmark_mse)
    return {
        "rmse": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "r2": float(r2_score(y_true, y_pred)),
        "oos_r2": float(oos_r2) if pd.notna(oos_r2) else np.nan,
        "sign_accuracy": float((np.sign(y_pred) == np.sign(y_true)).mean()),
        "actual_mean": float(np.mean(y_true)),
        "pred_mean": float(np.mean(y_pred)),
        "benchmark_mean": float(np.mean(benchmark_pred)),
    }


def split_day_segments(unique_days, gap_days=30):
    unique_days = pd.Index(sorted(pd.Index(unique_days).unique()))
    if len(unique_days) == 0:
        return []

    segments = []
    start = 0
    diffs = np.diff(unique_days.view("int64")) / (24 * 3600 * 1e9)
    for idx, gap in enumerate(diffs, start=1):
        if gap > gap_days:
            segments.append(unique_days[start:idx])
            start = idx
    segments.append(unique_days[start:])
    return segments


def allocate_splits_to_segments(
    segments,
    n_splits,
    min_train_days,
    purge_days,
    min_valid_days,
    embargo_days,
    min_prefix_days_later_segment,
):
    eligible = []
    for seg_idx, segment in enumerate(segments):
        required_days = (
            min_train_days + purge_days + min_valid_days
            if seg_idx == 0
            else min_prefix_days_later_segment + purge_days + min_valid_days
        )
        if len(segment) < required_days:
            continue

        segment_capacity = int(
            (len(segment) - required_days) // max(min_valid_days + embargo_days, 1)
        ) + 1
        if segment_capacity <= 0:
            continue

        eligible.append(
            {
                "segment_idx": seg_idx,
                "segment_len": len(segment),
                "capacity": segment_capacity,
            }
        )

    if not eligible:
        raise ValueError("No eligible day segments available for the requested split design.")

    allocation = {item["segment_idx"]: 0 for item in eligible}
    remaining = n_splits

    for item in eligible:
        if remaining <= 0:
            break
        allocation[item["segment_idx"]] += 1
        remaining -= 1

    while remaining > 0:
        best_choice = None
        for item in eligible:
            current = allocation[item["segment_idx"]]
            if current >= item["capacity"]:
                continue
            score = item["segment_len"] / (current + 1)
            if best_choice is None or score > best_choice[0]:
                best_choice = (score, item["segment_idx"])

        if best_choice is None:
            break

        allocation[best_choice[1]] += 1
        remaining -= 1

    return allocation


def build_segmented_day_splits(
    unique_days,
    n_splits,
    purge_days,
    embargo_days,
    min_train_days,
    min_valid_days,
    gap_days=30,
    min_prefix_days_later_segment=30,
):
    unique_days = pd.Index(sorted(pd.Index(unique_days).unique()))
    segments = split_day_segments(unique_days, gap_days=gap_days)
    allocation = allocate_splits_to_segments(
        segments=segments,
        n_splits=n_splits,
        min_train_days=min_train_days,
        purge_days=purge_days,
        min_valid_days=min_valid_days,
        embargo_days=embargo_days,
        min_prefix_days_later_segment=min_prefix_days_later_segment,
    )

    splits = []
    prior_days = []
    fold_id = 1

    for seg_idx, segment in enumerate(segments):
        n_segment_splits = allocation.get(seg_idx, 0)
        if n_segment_splits == 0:
            prior_days.extend(list(segment))
            continue

        prefix_days = min_train_days if seg_idx == 0 else min_prefix_days_later_segment
        valid_start_min = prefix_days + purge_days
        available_end = len(segment) - min_valid_days
        if available_end < valid_start_min:
            prior_days.extend(list(segment))
            continue

        raw_starts = (
            np.array([valid_start_min])
            if n_segment_splits == 1
            else np.linspace(valid_start_min, available_end, n_segment_splits, dtype=int)
        )
        raw_starts = np.maximum.accumulate(raw_starts)
        previous_valid_end = None

        for local_idx, start_pos in enumerate(raw_starts, start=1):
            if previous_valid_end is not None:
                start_pos = max(start_pos, previous_valid_end + 1 + embargo_days)

            remaining_after = n_segment_splits - local_idx
            max_end_pos = (
                len(segment) - 1
                if remaining_after == 0
                else len(segment) - remaining_after * (min_valid_days + embargo_days) - 1
            )
            next_start = raw_starts[local_idx] if local_idx < n_segment_splits else len(segment)
            end_pos = max(
                start_pos + min_valid_days - 1,
                next_start - 1 - embargo_days,
            )
            end_pos = min(end_pos, max_end_pos)

            train_seg_end = start_pos - purge_days - 1
            train_days = pd.Index(list(prior_days) + list(segment[: train_seg_end + 1]))
            valid_days = segment[start_pos : end_pos + 1]

            purge_start = segment[train_seg_end + 1] if train_seg_end + 1 < len(segment) else pd.NaT
            purge_end = segment[start_pos - 1] if start_pos - 1 < len(segment) else pd.NaT
            embargo_start = segment[end_pos + 1] if end_pos + 1 < len(segment) else pd.NaT
            embargo_end_pos = min(end_pos + embargo_days, len(segment) - 1)
            embargo_end = segment[embargo_end_pos] if end_pos + 1 < len(segment) else pd.NaT

            splits.append(
                {
                    "fold_id": fold_id,
                    "segment_idx": seg_idx,
                    "train_days": train_days,
                    "valid_days": valid_days,
                    "train_start_day": train_days.min(),
                    "train_end_day": train_days.max(),
                    "valid_start_day": valid_days.min(),
                    "valid_end_day": valid_days.max(),
                    "purge_start_day": purge_start,
                    "purge_end_day": purge_end,
                    "embargo_start_day": embargo_start,
                    "embargo_end_day": embargo_end,
                    "train_day_count": len(train_days),
                    "valid_day_count": len(valid_days),
                }
            )
            previous_valid_end = end_pos
            fold_id += 1

        prior_days.extend(list(segment))

    return splits


def make_cv_index_pairs(frame, split_records):
    frame = frame.sort_values(DATE_COL).reset_index(drop=True)
    index_pairs = []
    split_meta_rows = []
    for split in split_records:
        train_idx = frame.index[frame[DATE_COL].isin(split["train_days"])].to_numpy()
        valid_idx = frame.index[frame[DATE_COL].isin(split["valid_days"])].to_numpy()
        if len(train_idx) == 0 or len(valid_idx) == 0:
            continue

        index_pairs.append((train_idx, valid_idx))
        split_meta_rows.append(
            {
                "fold_id": split["fold_id"],
                "segment_idx": split["segment_idx"],
                "train_start_day": split["train_start_day"],
                "train_end_day": split["train_end_day"],
                "valid_start_day": split["valid_start_day"],
                "valid_end_day": split["valid_end_day"],
                "purge_start_day": split["purge_start_day"],
                "purge_end_day": split["purge_end_day"],
                "embargo_start_day": split["embargo_start_day"],
                "embargo_end_day": split["embargo_end_day"],
                "train_day_count": split["train_day_count"],
                "valid_day_count": split["valid_day_count"],
                "train_rows": len(train_idx),
                "valid_rows": len(valid_idx),
            }
        )
    return index_pairs, pd.DataFrame(split_meta_rows)


def choose_inner_cv(frame):
    unique_days = pd.Index(sorted(frame[DATE_COL].dropna().unique()))
    try:
        split_records = build_segmented_day_splits(
            unique_days=unique_days,
            n_splits=INNER_N_SPLITS,
            purge_days=INNER_PURGE_DAYS,
            embargo_days=INNER_EMBARGO_DAYS,
            min_train_days=INNER_MIN_TRAIN_DAYS,
            min_valid_days=INNER_MIN_VALID_DAYS,
            gap_days=GAP_SPLIT_DAYS,
            min_prefix_days_later_segment=MIN_PREFIX_DAYS_LATER_SEGMENT,
        )
        cv_pairs, split_df = make_cv_index_pairs(frame, split_records)
        if len(cv_pairs) >= 2:
            return cv_pairs, split_df, "segmented_purged"
    except ValueError:
        pass

    fallback_splits = min(INNER_N_SPLITS, max(frame[DATE_COL].nunique() - 1, 1))
    if fallback_splits < 2:
        raise ValueError("Not enough unique dates to build inner CV.")

    fallback = TimeSeriesSplit(n_splits=fallback_splits)
    fallback_pairs = list(fallback.split(frame))
    fallback_rows = []
    for idx, (train_idx, valid_idx) in enumerate(fallback_pairs, start=1):
        fallback_rows.append(
            {
                "fold_id": idx,
                "segment_idx": -1,
                "train_start_day": frame.iloc[train_idx][DATE_COL].min(),
                "train_end_day": frame.iloc[train_idx][DATE_COL].max(),
                "valid_start_day": frame.iloc[valid_idx][DATE_COL].min(),
                "valid_end_day": frame.iloc[valid_idx][DATE_COL].max(),
                "purge_start_day": pd.NaT,
                "purge_end_day": pd.NaT,
                "embargo_start_day": pd.NaT,
                "embargo_end_day": pd.NaT,
                "train_day_count": frame.iloc[train_idx][DATE_COL].nunique(),
                "valid_day_count": frame.iloc[valid_idx][DATE_COL].nunique(),
                "train_rows": len(train_idx),
                "valid_rows": len(valid_idx),
            }
        )
    return fallback_pairs, pd.DataFrame(fallback_rows), "time_series_fallback"


def build_model_bundle(model_name, search_profile):
    if model_name == "ridge":
        estimator = Ridge()
        if search_profile == "protocol":
            grid = {"model__alpha": [0.1, 1.0, 10.0]}
        elif search_profile == "targeted_followup":
            grid = {"model__alpha": [10.0, 30.0, 100.0, 300.0]}
        else:
            grid = {"model__alpha": [0.1, 1.0, 10.0, 100.0]}
    elif model_name == "random_forest":
        estimator = RandomForestRegressor(
            random_state=RANDOM_STATE,
            n_jobs=1,
        )
        if search_profile == "protocol":
            grid = {
                "model__n_estimators": [200],
                "model__max_depth": [3, 5],
                "model__min_samples_leaf": [1, 5],
                "model__max_features": ["sqrt", 0.5],
            }
        elif search_profile == "targeted_followup":
            grid = {
                "model__n_estimators": [200, 500],
                "model__max_depth": [2, 3, 5],
                "model__min_samples_leaf": [1, 5],
                "model__max_features": ["sqrt", 0.5],
            }
        else:
            grid = {
                "model__n_estimators": [200],
                "model__max_depth": [3, 5],
                "model__min_samples_leaf": [1, 5],
                "model__max_features": ["sqrt", 0.5],
            }
    elif model_name == "xgboost_regressor":
        estimator = XGBRegressor(
            objective="reg:squarederror",
            eval_metric="rmse",
            random_state=RANDOM_STATE,
            n_jobs=1,
            tree_method="hist",
            verbosity=0,
        )
        if search_profile == "protocol":
            grid = {
                "model__n_estimators": [200],
                "model__max_depth": [2, 3],
                "model__learning_rate": [0.03, 0.08],
                "model__min_child_weight": [1, 5],
                "model__subsample": [0.8],
                "model__colsample_bytree": [0.8],
            }
        elif search_profile == "targeted_followup":
            grid = {
                "model__n_estimators": [200, 500],
                "model__max_depth": [1, 2, 3],
                "model__learning_rate": [0.01, 0.03],
                "model__min_child_weight": [5, 10],
                "model__subsample": [0.8],
                "model__colsample_bytree": [0.8],
            }
        else:
            grid = {
                "model__n_estimators": [200],
                "model__max_depth": [2, 3],
                "model__learning_rate": [0.03, 0.08],
                "model__min_child_weight": [1, 5],
                "model__subsample": [0.8],
                "model__colsample_bytree": [0.8],
            }
    else:
        raise ValueError(f"Unknown model_name: {model_name}")

    return estimator, grid


def make_model_pipeline(features, model_name):
    estimator, grid = build_model_bundle(model_name=model_name, search_profile=SEARCH_PROFILE)
    preprocessor = ColumnTransformer(
        transformers=[
            (
                "num",
                Pipeline(
                    steps=[
                        ("imputer", SimpleImputer(strategy="median")),
                        ("scaler", StandardScaler()),
                    ]
                ),
                features,
            ),
        ],
        remainder="drop",
    )
    pipeline = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", estimator),
        ]
    )
    return pipeline, grid


def extract_model_importance(estimator, features, variant_key, spec_name, model_name):
    fitted_model = estimator.named_steps["model"]
    if hasattr(fitted_model, "coef_"):
        values = fitted_model.coef_
        importance_type = "coefficient"
    elif hasattr(fitted_model, "feature_importances_"):
        values = fitted_model.feature_importances_
        importance_type = "feature_importance"
    else:
        values = np.repeat(np.nan, len(features))
        importance_type = "not_available"

    return pd.DataFrame(
        {
            "variant_key": variant_key,
            "model_name": model_name,
            "spec_name": spec_name,
            "feature_name": features,
            "importance_value": values,
            "abs_importance_value": np.abs(values),
            "importance_type": importance_type,
        }
    )


def flatten_summary_columns(frame):
    flat_cols = []
    for col in frame.columns:
        if isinstance(col, tuple):
            flat_cols.append("_".join([str(part) for part in col if part]).strip("_"))
        else:
            flat_cols.append(str(col))
    out = frame.copy()
    out.columns = flat_cols
    return out.reset_index()

In [4]:
df = pd.read_csv(DATA_PATH)
df[DATE_COL] = pd.to_datetime(df[DATE_COL]).dt.normalize()
df = df.sort_values(["variant_key", DATE_COL]).reset_index(drop=True)

if not ACTIVE_VARIANTS:
    ACTIVE_VARIANTS = sorted(df["variant_key"].dropna().unique().tolist())
missing_variants = [key for key in ACTIVE_VARIANTS if key not in df["variant_key"].dropna().unique()]
if missing_variants:
    raise ValueError(f"Missing active variants in compare input: {missing_variants}")

active_feature_columns = sorted(
    set(
        feature
        for feature_cols in MODEL_SPECS.values()
        for feature in feature_cols
    )
)
required_columns = sorted(
    set([TARGET_COL, DATE_COL, "stage2_split", "variant_key"] + active_feature_columns)
)
missing_required = [col for col in required_columns if col not in df.columns]
if missing_required:
    raise ValueError(f"Missing required columns: {missing_required}")

main_df = df[df["stage2_split"].isin([TRAIN_LABEL, HOLDOUT_LABEL])].copy()
main_df = main_df.dropna(subset=[TARGET_COL]).sort_values(["variant_key", DATE_COL]).reset_index(drop=True)

print(f"Loaded {len(df):,} rows and {df.shape[1]} columns from {DATA_PATH}")
print(f"Main sample rows with non-null target: {len(main_df):,}")
print("Search profile:", SEARCH_PROFILE)

Loaded 2,293 rows and 52 columns from /Users/horace/Coding/ML finance/project/main_rerun/artifacts/stage2/stage2_daily_features_long_v2_main_rerun.csv
Main sample rows with non-null target: 1,204
Search profile: targeted_followup


In [5]:
split_summary = (
    main_df.groupby(["variant_key", "stage2_split"])
    .agg(
        n_rows=(DATE_COL, "size"),
        n_unique_dates=(DATE_COL, "nunique"),
        first_date=(DATE_COL, "min"),
        last_date=(DATE_COL, "max"),
        target_mean=(TARGET_COL, "mean"),
        target_std=(TARGET_COL, "std"),
    )
    .reset_index()
    .sort_values(["variant_key", "stage2_split"])
)
display(split_summary)

missing_summary = (
    main_df[CORE_CONTROL_FEATURES + P_TACO_V2_CORE_FEATURES + P_TACO_V2_INTERACTION_FEATURES + [TARGET_COL]]
    .isna()
    .mean()
    .sort_values(ascending=False)
    .rename("missing_rate")
    .to_frame()
)
display(missing_summary)

,variant_key,stage2_split,n_rows,n_unique_dates,first_date,last_date,target_mean,target_std
0,linear_svm__sigmoid,primary_holdout,74,74,2025-07-15,2025-10-24,0.002761,0.014441
1,linear_svm__sigmoid,train_pool,1130,1130,2017-01-20,2025-07-14,0.000570,0.008912


,missing_rate
dxy_ret_1d_lag1,0.038206
spx_ret_1d_lag1,0.018272
spx_drawdown_5d_lag1,0.018272
vix_change_5d_lag1,0.008306
real_yield_change_5d_lag1,0.000000
vix_ma_5d_lag1,0.000000
gold_spx_corr_20d_lag1,0.000000
p_taco_any,0.000000
p_taco_top1,0.000000
p_taco_tail_010,0.000000


In [6]:
outer_unique_days = pd.Index(
    sorted(main_df.loc[main_df["stage2_split"] == TRAIN_LABEL, DATE_COL].dropna().unique())
)
outer_split_records = build_segmented_day_splits(
    unique_days=outer_unique_days,
    n_splits=OUTER_N_SPLITS,
    purge_days=OUTER_PURGE_DAYS,
    embargo_days=OUTER_EMBARGO_DAYS,
    min_train_days=OUTER_MIN_TRAIN_DAYS,
    min_valid_days=OUTER_MIN_VALID_DAYS,
    gap_days=GAP_SPLIT_DAYS,
    min_prefix_days_later_segment=MIN_PREFIX_DAYS_LATER_SEGMENT,
)
_, outer_splits_df = make_cv_index_pairs(
    frame=main_df[main_df["variant_key"] == ACTIVE_VARIANTS[0]].copy(),
    split_records=outer_split_records,
)
display(outer_splits_df)

,fold_id,segment_idx,train_start_day,train_end_day,valid_start_day,valid_end_day,purge_start_day,purge_end_day,embargo_start_day,embargo_end_day,train_day_count,valid_day_count,train_rows,valid_rows
0,1,0,2017-01-20,2017-10-04,2017-10-09,2018-10-16,2017-10-05,2017-10-06,2018-10-17,2018-10-23,180,259,180,259
1,2,0,2017-01-20,2018-10-19,2018-10-24,2019-11-01,2018-10-22,2018-10-23,2019-11-04,2019-11-08,444,259,444,259
2,3,0,2017-01-20,2019-11-06,2019-11-11,2020-11-17,2019-11-07,2019-11-08,2020-11-18,2020-11-24,708,259,708,259
3,4,0,2017-01-20,2020-11-20,2020-11-25,2021-01-08,2020-11-23,2020-11-24,NaT,NaT,972,30,972,30
4,5,1,2017-01-20,2025-02-28,2025-03-05,2025-07-07,2025-03-03,2025-03-04,2025-07-08,2025-07-14,1034,89,1034,89


In [7]:
fig, ax = plt.subplots(figsize=(16, 6))
for _, row in outer_splits_df.iterrows():
    ax.plot([row["train_start_day"], row["train_end_day"]], [row["fold_id"], row["fold_id"]], color="#457b9d", linewidth=6, solid_capstyle="butt")
    ax.plot([row["valid_start_day"], row["valid_end_day"]], [row["fold_id"], row["fold_id"]], color="#e76f51", linewidth=6, solid_capstyle="butt")
ax.set_title("Stage 2 Outer Splits: Train (blue) vs Validation (red)")
ax.set_xlabel("date")
ax.set_ylabel("fold_id")
plt.tight_layout()
plt.show()

In [8]:
outer_fold_metric_rows = []
outer_prediction_frames = []
best_param_rows = []
holdout_metric_rows = []
holdout_prediction_frames = []
holdout_importance_frames = []

for variant_key in ACTIVE_VARIANTS:
    variant_df = (
        main_df[main_df["variant_key"] == variant_key]
        .sort_values(DATE_COL)
        .reset_index(drop=True)
    )
    train_pool_df = variant_df[variant_df["stage2_split"] == TRAIN_LABEL].copy().reset_index(drop=True)
    holdout_df = variant_df[variant_df["stage2_split"] == HOLDOUT_LABEL].copy().reset_index(drop=True)

    print(
        f"{variant_key}: train_pool={len(train_pool_df):,} rows / {train_pool_df[DATE_COL].nunique():,} days, "
        f"holdout={len(holdout_df):,} rows / {holdout_df[DATE_COL].nunique():,} days"
    )

    for model_name in ACTIVE_MODELS:
        print(f"  - model {model_name}")

        for spec_name, feature_cols in MODEL_SPECS.items():
            print(f"    * spec {spec_name}")

            for outer_split in outer_split_records:
                outer_train = train_pool_df[train_pool_df[DATE_COL].isin(outer_split["train_days"])].copy().reset_index(drop=True)
                outer_valid = train_pool_df[train_pool_df[DATE_COL].isin(outer_split["valid_days"])].copy().reset_index(drop=True)

                inner_cv_pairs, inner_splits_df, inner_strategy = choose_inner_cv(outer_train)
                pipeline, param_grid = make_model_pipeline(feature_cols, model_name=model_name)
                search = GridSearchCV(
                    estimator=pipeline,
                    param_grid=param_grid,
                    scoring="neg_root_mean_squared_error",
                    cv=inner_cv_pairs,
                    refit=True,
                    n_jobs=None,
                )
                search.fit(outer_train[feature_cols], outer_train[TARGET_COL])

                valid_pred = search.best_estimator_.predict(outer_valid[feature_cols])
                outer_benchmark_pred = np.repeat(
                    outer_train[TARGET_COL].mean(),
                    len(outer_valid),
                )
                fold_metrics = compute_regression_metrics(
                    outer_valid[TARGET_COL].to_numpy(),
                    valid_pred,
                    benchmark_pred=outer_benchmark_pred,
                )
                fold_metrics.update(
                    {
                        "variant_key": variant_key,
                        "model_name": model_name,
                        "spec_name": spec_name,
                        "fold_id": outer_split["fold_id"],
                        "segment_idx": outer_split["segment_idx"],
                        "inner_cv_strategy": inner_strategy,
                        "best_params_json": json.dumps(search.best_params_, ensure_ascii=False),
                        "best_inner_rmse": float(-search.best_score_),
                        "train_start_day": outer_split["train_start_day"],
                        "train_end_day": outer_split["train_end_day"],
                        "valid_start_day": outer_split["valid_start_day"],
                        "valid_end_day": outer_split["valid_end_day"],
                        "train_day_count": outer_split["train_day_count"],
                        "valid_day_count": outer_split["valid_day_count"],
                        "train_rows": len(outer_train),
                        "valid_rows": len(outer_valid),
                    }
                )
                outer_fold_metric_rows.append(fold_metrics)

                outer_prediction_frames.append(
                    pd.DataFrame(
                        {
                            "date": outer_valid[DATE_COL].to_numpy(),
                            "variant_key": variant_key,
                            "model_name": model_name,
                            "spec_name": spec_name,
                            "fold_id": outer_split["fold_id"],
                            "y_true": outer_valid[TARGET_COL].to_numpy(),
                            "y_pred": valid_pred,
                        }
                    )
                )

                best_param_rows.append(
                    {
                        "variant_key": variant_key,
                        "model_name": model_name,
                        "spec_name": spec_name,
                        "fold_id": outer_split["fold_id"],
                        "segment_idx": outer_split["segment_idx"],
                        "best_params_json": json.dumps(search.best_params_, ensure_ascii=False),
                        "best_inner_rmse": float(-search.best_score_),
                        "inner_cv_strategy": inner_strategy,
                        "feature_list_json": json.dumps(feature_cols, ensure_ascii=False),
                    }
                )

            full_inner_pairs, full_inner_splits_df, full_inner_strategy = choose_inner_cv(train_pool_df)
            full_pipeline, full_param_grid = make_model_pipeline(feature_cols, model_name=model_name)
            full_search = GridSearchCV(
                estimator=full_pipeline,
                param_grid=full_param_grid,
                scoring="neg_root_mean_squared_error",
                cv=full_inner_pairs,
                refit=True,
                n_jobs=None,
            )
            full_search.fit(train_pool_df[feature_cols], train_pool_df[TARGET_COL])
            holdout_pred = full_search.best_estimator_.predict(holdout_df[feature_cols])
            holdout_benchmark_pred = np.repeat(
                train_pool_df[TARGET_COL].mean(),
                len(holdout_df),
            )
            holdout_metrics = compute_regression_metrics(
                holdout_df[TARGET_COL].to_numpy(),
                holdout_pred,
                benchmark_pred=holdout_benchmark_pred,
            )
            holdout_metrics.update(
                {
                    "variant_key": variant_key,
                    "model_name": model_name,
                    "spec_name": spec_name,
                    "best_params_json": json.dumps(full_search.best_params_, ensure_ascii=False),
                    "best_inner_rmse": float(-full_search.best_score_),
                    "inner_cv_strategy": full_inner_strategy,
                    "n_obs": len(holdout_df),
                }
            )
            holdout_metric_rows.append(holdout_metrics)

            holdout_prediction_frames.append(
                pd.DataFrame(
                    {
                        "date": holdout_df[DATE_COL].to_numpy(),
                        "variant_key": variant_key,
                        "model_name": model_name,
                        "spec_name": spec_name,
                        "y_true": holdout_df[TARGET_COL].to_numpy(),
                        "y_pred": holdout_pred,
                    }
                )
            )

            holdout_importance_frames.append(
                extract_model_importance(
                    estimator=full_search.best_estimator_,
                    features=feature_cols,
                    variant_key=variant_key,
                    spec_name=spec_name,
                    model_name=model_name,
                )
            )

outer_fold_metrics_df = pd.DataFrame(outer_fold_metric_rows)
outer_predictions_df = pd.concat(outer_prediction_frames, ignore_index=True)
best_params_df = pd.DataFrame(best_param_rows)
holdout_metrics_df = pd.DataFrame(holdout_metric_rows).sort_values(["variant_key", "model_name", "spec_name"]).reset_index(drop=True)
holdout_predictions_df = pd.concat(holdout_prediction_frames, ignore_index=True)
holdout_importances_df = pd.concat(holdout_importance_frames, ignore_index=True)

outer_summary_df = flatten_summary_columns(
    outer_fold_metrics_df.groupby(["variant_key", "model_name", "spec_name"])[["rmse", "mae", "r2", "oos_r2", "sign_accuracy"]]
    .agg(["mean", "std"])
)

display(outer_summary_df.sort_values(["variant_key", "model_name", "rmse_mean"]))
display(holdout_metrics_df[["variant_key", "model_name", "spec_name", "rmse", "mae", "r2", "oos_r2", "sign_accuracy"]])

linear_svm__sigmoid: train_pool=1,130 rows / 1,130 days, holdout=74 rows / 74 days
  - model ridge
    * spec macro_only


    * spec two_stage_v2_core


  - model random_forest
    * spec macro_only


    * spec two_stage_v2_core


  - model xgboost_regressor
    * spec macro_only


    * spec two_stage_v2_core


,variant_key,model_name,spec_name,rmse_mean,rmse_std,mae_mean,mae_std,r2_mean,r2_std,oos_r2_mean,oos_r2_std,sign_accuracy_mean,sign_accuracy_std
0,linear_svm__sigmoid,random_forest,macro_only,0.009833,0.002811,0.007209,0.002058,-0.010833,0.029858,-0.005132,0.029369,0.524743,0.052815
1,linear_svm__sigmoid,random_forest,two_stage_v2_core,0.009860,0.002743,0.007238,0.002011,-0.021578,0.052347,-0.015805,0.051590,0.520812,0.055863
2,linear_svm__sigmoid,ridge,macro_only,0.010032,0.002701,0.007324,0.001997,-0.067653,0.121520,-0.061609,0.120490,0.534148,0.046648
3,linear_svm__sigmoid,ridge,two_stage_v2_core,0.010054,0.002727,0.007341,0.002018,-0.071204,0.119928,-0.065135,0.118850,0.525862,0.051203
5,linear_svm__sigmoid,xgboost_regressor,two_stage_v2_core,0.009854,0.002793,0.007244,0.002053,-0.016860,0.037197,-0.011123,0.036678,0.522426,0.053049
4,linear_svm__sigmoid,xgboost_regressor,macro_only,0.009865,0.002779,0.007247,0.002033,-0.020194,0.039756,-0.014428,0.038862,0.514774,0.053525


,variant_key,model_name,spec_name,rmse,mae,r2,oos_r2,sign_accuracy
0,linear_svm__sigmoid,random_forest,macro_only,0.014471,0.009331,-0.017907,0.005295,0.635135
1,linear_svm__sigmoid,random_forest,two_stage_v2_core,0.014453,0.009367,-0.015483,0.007664,0.648649
2,linear_svm__sigmoid,ridge,macro_only,0.014575,0.009362,-0.032660,-0.009122,0.621622
3,linear_svm__sigmoid,ridge,two_stage_v2_core,0.014564,0.009363,-0.031046,-0.007545,0.608108
4,linear_svm__sigmoid,xgboost_regressor,macro_only,0.014477,0.009362,-0.018869,0.004355,0.621622
5,linear_svm__sigmoid,xgboost_regressor,two_stage_v2_core,0.014484,0.009378,-0.019855,0.003391,0.635135


In [9]:
holdout_compare = flatten_summary_columns(
    holdout_metrics_df.pivot(index=["variant_key", "model_name"], columns="spec_name", values=["rmse", "mae", "r2", "oos_r2", "sign_accuracy"])
)
has_macro_baseline = "macro_only" in MODEL_SPECS
for spec_name in [name for name in MODEL_SPECS if name.startswith("two_stage_")]:
    if has_macro_baseline and f"rmse_{spec_name}" in holdout_compare.columns:
        holdout_compare[f"delta_rmse_{spec_name}_minus_macro"] = (
            holdout_compare[f"rmse_{spec_name}"] - holdout_compare["rmse_macro_only"]
        )
        holdout_compare[f"delta_mae_{spec_name}_minus_macro"] = (
            holdout_compare[f"mae_{spec_name}"] - holdout_compare["mae_macro_only"]
        )
        holdout_compare[f"delta_r2_{spec_name}_minus_macro"] = (
            holdout_compare[f"r2_{spec_name}"] - holdout_compare["r2_macro_only"]
        )
        holdout_compare[f"delta_oos_r2_{spec_name}_minus_macro"] = (
            holdout_compare[f"oos_r2_{spec_name}"] - holdout_compare["oos_r2_macro_only"]
        )
        holdout_compare[f"delta_sign_acc_{spec_name}_minus_macro"] = (
            holdout_compare[f"sign_accuracy_{spec_name}"] - holdout_compare["sign_accuracy_macro_only"]
        )
display(holdout_compare.sort_values(["variant_key", "model_name"]))

outer_compare = flatten_summary_columns(
    outer_summary_df.pivot(index=["variant_key", "model_name"], columns="spec_name", values=["rmse_mean", "mae_mean", "r2_mean", "oos_r2_mean", "sign_accuracy_mean"])
)
display(outer_compare.sort_values(["variant_key", "model_name"]))

,variant_key,model_name,rmse_macro_only,rmse_two_stage_v2_core,mae_macro_only,mae_two_stage_v2_core,r2_macro_only,r2_two_stage_v2_core,oos_r2_macro_only,oos_r2_two_stage_v2_core,sign_accuracy_macro_only,sign_accuracy_two_stage_v2_core,delta_rmse_two_stage_v2_core_minus_macro,delta_mae_two_stage_v2_core_minus_macro,delta_r2_two_stage_v2_core_minus_macro,delta_oos_r2_two_stage_v2_core_minus_macro,delta_sign_acc_two_stage_v2_core_minus_macro
0,linear_svm__sigmoid,random_forest,0.014471,0.014453,0.009331,0.009367,-0.017907,-0.015483,0.005295,0.007664,0.635135,0.648649,-0.000017,3.525398e-05,0.002423,0.002368,0.013514
1,linear_svm__sigmoid,ridge,0.014575,0.014564,0.009362,0.009363,-0.032660,-0.031046,-0.009122,-0.007545,0.621622,0.608108,-0.000011,4.285205e-07,0.001614,0.001577,-0.013514
2,linear_svm__sigmoid,xgboost_regressor,0.014477,0.014484,0.009362,0.009378,-0.018869,-0.019855,0.004355,0.003391,0.621622,0.635135,0.000007,1.601071e-05,-0.000987,-0.000964,0.013514


,variant_key,model_name,rmse_mean_macro_only,rmse_mean_two_stage_v2_core,mae_mean_macro_only,mae_mean_two_stage_v2_core,r2_mean_macro_only,r2_mean_two_stage_v2_core,oos_r2_mean_macro_only,oos_r2_mean_two_stage_v2_core,sign_accuracy_mean_macro_only,sign_accuracy_mean_two_stage_v2_core
0,linear_svm__sigmoid,random_forest,0.009833,0.009860,0.007209,0.007238,-0.010833,-0.021578,-0.005132,-0.015805,0.524743,0.520812
1,linear_svm__sigmoid,ridge,0.010032,0.010054,0.007324,0.007341,-0.067653,-0.071204,-0.061609,-0.065135,0.534148,0.525862
2,linear_svm__sigmoid,xgboost_regressor,0.009865,0.009854,0.007247,0.007244,-0.020194,-0.016860,-0.014428,-0.011123,0.514774,0.522426


In [10]:
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

sns.barplot(
    data=outer_summary_df,
    x="model_name",
    y="rmse_mean",
    hue="spec_name",
    ax=axes[0],
    palette="deep",
)
axes[0].set_title("Outer CV RMSE by Model")
axes[0].set_xlabel("model_name")
axes[0].set_ylabel("rmse_mean")

sns.barplot(
    data=holdout_metrics_df,
    x="model_name",
    y="rmse",
    hue="spec_name",
    ax=axes[1],
    palette="muted",
)
axes[1].set_title(f"{HOLDOUT_DISPLAY_TITLE} RMSE by Model")
axes[1].set_xlabel("model_name")
axes[1].set_ylabel("rmse")

plt.tight_layout()
plt.show()

In [11]:
takeaway_rows = []
for variant_key in ACTIVE_VARIANTS:
    for model_name in ACTIVE_MODELS:
        outer_slice = outer_summary_df[
            (outer_summary_df["variant_key"] == variant_key)
            & (outer_summary_df["model_name"] == model_name)
        ].set_index("spec_name")
        hold_slice = holdout_metrics_df[
            (holdout_metrics_df["variant_key"] == variant_key)
            & (holdout_metrics_df["model_name"] == model_name)
        ].set_index("spec_name")

        best_outer_spec = outer_slice["rmse_mean"].idxmin()
        best_holdout_spec = hold_slice["rmse"].idxmin()
        takeaway_row = {
            "variant_key": variant_key,
            "model_name": model_name,
            "best_outer_spec": best_outer_spec,
            "best_holdout_spec": best_holdout_spec,
        }
        if "macro_only" in outer_slice.index and "macro_only" in hold_slice.index:
            takeaway_row["outer_delta_best_minus_macro"] = float(
                outer_slice.loc[best_outer_spec, "rmse_mean"] - outer_slice.loc["macro_only", "rmse_mean"]
            )
            takeaway_row["holdout_delta_best_minus_macro"] = float(
                hold_slice.loc[best_holdout_spec, "rmse"] - hold_slice.loc["macro_only", "rmse"]
            )
        else:
            takeaway_row["outer_delta_best_minus_macro"] = np.nan
            takeaway_row["holdout_delta_best_minus_macro"] = np.nan
        takeaway_rows.append(takeaway_row)
takeaway_df = pd.DataFrame(takeaway_rows).sort_values(["variant_key", "holdout_delta_best_minus_macro"])
display(takeaway_df)

,variant_key,model_name,best_outer_spec,best_holdout_spec,outer_delta_best_minus_macro,holdout_delta_best_minus_macro
1,linear_svm__sigmoid,random_forest,macro_only,two_stage_v2_core,0.000000,-0.000017
0,linear_svm__sigmoid,ridge,macro_only,two_stage_v2_core,0.000000,-0.000011
2,linear_svm__sigmoid,xgboost_regressor,two_stage_v2_core,macro_only,-0.000012,0.000000


In [12]:
outer_splits_df.to_csv(RESULTS_DIR / "stage2_nested_cv_model_compare_outer_splits.csv", index=False)
best_params_df.to_csv(RESULTS_DIR / "stage2_nested_cv_model_compare_best_params_by_fold.csv", index=False)
outer_fold_metrics_df.to_csv(RESULTS_DIR / "stage2_nested_cv_model_compare_outer_fold_metrics.csv", index=False)
outer_summary_df.to_csv(RESULTS_DIR / "stage2_nested_cv_model_compare_outer_summary_metrics.csv", index=False)
outer_predictions_df.to_csv(RESULTS_DIR / "stage2_nested_cv_model_compare_outer_predictions.csv", index=False)
holdout_metrics_df.to_csv(RESULTS_DIR / "stage2_nested_cv_model_compare_holdout_metrics.csv", index=False)
holdout_predictions_df.to_csv(RESULTS_DIR / "stage2_nested_cv_model_compare_holdout_predictions.csv", index=False)
holdout_importances_df.to_csv(RESULTS_DIR / "stage2_nested_cv_model_compare_holdout_importances.csv", index=False)
holdout_compare.to_csv(RESULTS_DIR / "stage2_nested_cv_model_compare_holdout_compare.csv", index=False)

print("Saved model-compare artifacts:")
for path in sorted(RESULTS_DIR.glob("stage2_nested_cv_model_compare_*.csv")):
    print(path)

Saved model-compare artifacts:
/Users/horace/Coding/ML finance/project/main_rerun/artifacts/stage2/final_compare_targeted_followup/stage2_nested_cv_model_compare_best_params_by_fold.csv
/Users/horace/Coding/ML finance/project/main_rerun/artifacts/stage2/final_compare_targeted_followup/stage2_nested_cv_model_compare_holdout_compare.csv
/Users/horace/Coding/ML finance/project/main_rerun/artifacts/stage2/final_compare_targeted_followup/stage2_nested_cv_model_compare_holdout_importances.csv
/Users/horace/Coding/ML finance/project/main_rerun/artifacts/stage2/final_compare_targeted_followup/stage2_nested_cv_model_compare_holdout_metrics.csv
/Users/horace/Coding/ML finance/project/main_rerun/artifacts/stage2/final_compare_targeted_followup/stage2_nested_cv_model_compare_holdout_predictions.csv
/Users/horace/Coding/ML finance/project/main_rerun/artifacts/stage2/final_compare_targeted_followup/stage2_nested_cv_model_compare_outer_fold_metrics.csv
/Users/horace/Coding/ML finance/project/main_rer

# Main Rerun Stage 2 Final Model SHAP

- 对 `main_rerun` 当前正式 `Stage 2` 最优模型做一次 full-train refit
- 在 `primary_holdout` 上输出一张 SHAP summary 图


In [ ]:
from pathlib import Path
import json
import os
import warnings

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

try:
    import shap
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "This notebook requires the `shap` package. Install it in the active environment before running."
    ) from exc

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="talk")

CWD = Path.cwd().resolve()
PROJECT_ROOT = None
for candidate in [CWD, *CWD.parents]:
    if (candidate / "main_rerun" / "artifacts" / "stage2").exists():
        PROJECT_ROOT = candidate
        break
if PROJECT_ROOT is None:
    raise RuntimeError("Could not locate project root containing `main_rerun/artifacts/stage2`.")

os.environ.setdefault("MPLCONFIGDIR", str(PROJECT_ROOT / ".mpl-cache"))

MAIN_RERUN_ROOT = PROJECT_ROOT / "main_rerun"
STAGE2_ARTIFACT_DIR = MAIN_RERUN_ROOT / "artifacts" / "stage2"
FINAL_COMPARE_DIR = STAGE2_ARTIFACT_DIR / "final_compare"
OUT_DIR = FINAL_COMPARE_DIR / "shap"
OUT_DIR.mkdir(parents=True, exist_ok=True)

DATA_PATH = STAGE2_ARTIFACT_DIR / "stage2_daily_features_long_v2_main_rerun.csv"
SELECTED_MODEL_PATH = FINAL_COMPARE_DIR / "stage2_nested_cv_model_compare_selected_model.csv"
HOLDOUT_METRICS_PATH = FINAL_COMPARE_DIR / "stage2_nested_cv_model_compare_holdout_metrics.csv"

TARGET_COL = "gold_ret_1d"
DATE_COL = "date"
TRAIN_LABEL = "train_pool"
HOLDOUT_LABEL = "primary_holdout"
RANDOM_STATE = 42

print("PROJECT_ROOT:", PROJECT_ROOT)
print("STAGE2_ARTIFACT_DIR:", STAGE2_ARTIFACT_DIR)
print("FINAL_COMPARE_DIR:", FINAL_COMPARE_DIR)
print("OUT_DIR:", OUT_DIR)


In [ ]:
P_TACO_V2_CORE_FEATURES = [
    "p_taco_any",
    "p_taco_top1",
    "p_taco_tail_010",
    "p_taco_any_decay_2d",
]

CORE_CONTROL_FEATURES = [
    "dxy_ret_1d_lag1",
    "real_yield_change_5d_lag1",
    "vix_ma_5d_lag1",
    "vix_change_5d_lag1",
    "spx_ret_1d_lag1",
    "spx_drawdown_5d_lag1",
    "gold_spx_corr_20d_lag1",
]

MODEL_SPEC_CATALOG = {
    "two_stage_v2_core": P_TACO_V2_CORE_FEATURES + CORE_CONTROL_FEATURES,
}


def parse_best_param_string(text):
    data = json.loads(text)
    return {key.replace("model__", ""): value for key, value in data.items()}


def coerce_matrix(matrix):
    return matrix.toarray() if hasattr(matrix, "toarray") else np.asarray(matrix)


def coerce_shap_values(shap_output):
    if hasattr(shap_output, "values"):
        values = shap_output.values
    else:
        values = shap_output
    if isinstance(values, list):
        values = values[-1]
    values = np.asarray(values)
    if values.ndim == 3:
        values = values[:, :, -1]
    return values


def build_pipeline(features, model_params):
    return Pipeline(
        steps=[
            (
                "preprocessor",
                ColumnTransformer(
                    transformers=[
                        (
                            "num",
                            Pipeline(
                                steps=[
                                    ("imputer", SimpleImputer(strategy="median")),
                                    ("scaler", StandardScaler()),
                                ]
                            ),
                            features,
                        )
                    ],
                    remainder="drop",
                ),
            ),
            (
                "model",
                RandomForestRegressor(
                    random_state=RANDOM_STATE,
                    n_jobs=1,
                    **model_params,
                ),
            ),
        ]
    )


def save_summary_plot(shap_values, feature_matrix, feature_names, out_path):
    plt.figure(figsize=(12, 8))
    shap.summary_plot(
        shap_values,
        feature_matrix,
        feature_names=feature_names,
        show=False,
        max_display=len(feature_names),
    )
    plt.title("Main Rerun Stage 2 Final Model SHAP Summary")
    plt.tight_layout()
    plt.savefig(out_path, dpi=220, bbox_inches="tight")
    plt.close()


In [ ]:
selected_model_df = pd.read_csv(SELECTED_MODEL_PATH)
if len(selected_model_df) != 1:
    raise ValueError(
        f"Expected exactly one selected Stage 2 final model row, got {len(selected_model_df)}."
    )

selected_row = selected_model_df.iloc[0]
variant_key = selected_row["variant_key"]
model_name = selected_row["model_name"]
spec_name = selected_row["spec_name"]

if model_name != "random_forest":
    raise ValueError(f"Selected model is `{model_name}`, not `random_forest`.")
if spec_name not in MODEL_SPEC_CATALOG:
    raise ValueError(f"Unsupported spec_name `{spec_name}`.")

feature_cols = MODEL_SPEC_CATALOG[spec_name]

holdout_metrics_df = pd.read_csv(HOLDOUT_METRICS_PATH)
selected_holdout_row = holdout_metrics_df[
    (holdout_metrics_df["variant_key"] == variant_key)
    & (holdout_metrics_df["model_name"] == model_name)
    & (holdout_metrics_df["spec_name"] == spec_name)
].copy()
if len(selected_holdout_row) != 1:
    raise ValueError("Expected exactly one selected holdout metrics row for final model.")
selected_holdout_row = selected_holdout_row.iloc[0]

model_params = parse_best_param_string(selected_holdout_row["best_params_json"])
selected_holdout_row[["variant_key", "model_name", "spec_name", "best_params_json", "rmse", "mae", "oos_r2", "sign_accuracy"]]


In [ ]:
with (OUT_DIR / "stage2_final_selected_model_params.json").open("w", encoding="utf-8") as handle:
    json.dump(model_params, handle, indent=2, ensure_ascii=False)

df = pd.read_csv(DATA_PATH)
df[DATE_COL] = pd.to_datetime(df[DATE_COL]).dt.normalize()
df = df[
    (df["variant_key"] == variant_key)
    & (df["stage2_split"].isin([TRAIN_LABEL, HOLDOUT_LABEL]))
].copy()
df = df.dropna(subset=[TARGET_COL]).sort_values(DATE_COL).reset_index(drop=True)

missing_features = [col for col in feature_cols if col not in df.columns]
if missing_features:
    raise ValueError(f"Missing required features: {missing_features}")

train_df = df[df["stage2_split"] == TRAIN_LABEL].copy().reset_index(drop=True)
holdout_df = df[df["stage2_split"] == HOLDOUT_LABEL].copy().reset_index(drop=True)
if train_df.empty or holdout_df.empty:
    raise ValueError("Train pool or holdout slice is empty; cannot fit SHAP model.")

print("variant_key:", variant_key)
print("model_name:", model_name)
print("spec_name:", spec_name)
print("train_pool rows:", len(train_df))
print("primary_holdout rows:", len(holdout_df))
print("feature count:", len(feature_cols))


In [ ]:
pipeline = build_pipeline(features=feature_cols, model_params=model_params)
pipeline.fit(train_df[feature_cols], train_df[TARGET_COL])

holdout_pred = pipeline.predict(holdout_df[feature_cols])
holdout_prediction_df = holdout_df[[DATE_COL, TARGET_COL]].copy()
holdout_prediction_df["variant_key"] = variant_key
holdout_prediction_df["model_name"] = model_name
holdout_prediction_df["spec_name"] = spec_name
holdout_prediction_df["y_pred"] = holdout_pred
holdout_prediction_df.to_csv(OUT_DIR / "stage2_final_selected_model_holdout_predictions.csv", index=False)

holdout_prediction_df.head()


In [ ]:
preprocessor = pipeline.named_steps["preprocessor"]
model = pipeline.named_steps["model"]
holdout_matrix = coerce_matrix(preprocessor.transform(holdout_df[feature_cols]))

explainer = shap.TreeExplainer(model)
shap_values = coerce_shap_values(explainer.shap_values(holdout_matrix))

summary_plot_path = OUT_DIR / "stage2_final_selected_model_holdout_shap_summary.png"
save_summary_plot(
    shap_values=shap_values,
    feature_matrix=holdout_matrix,
    feature_names=feature_cols,
    out_path=summary_plot_path,
)

print("Saved summary plot to:", summary_plot_path)


In [ ]:
shap_values_df = pd.DataFrame(shap_values, columns=feature_cols)
shap_values_df.insert(0, DATE_COL, holdout_df[DATE_COL].dt.strftime("%Y-%m-%d").to_numpy())
shap_values_df.insert(1, TARGET_COL, holdout_df[TARGET_COL].to_numpy())
shap_values_df.insert(2, "y_pred", holdout_pred)
shap_values_df.to_csv(OUT_DIR / "stage2_final_selected_model_holdout_shap_values.csv", index=False)

feature_value_df = holdout_df[[DATE_COL] + feature_cols].copy()
feature_value_df[DATE_COL] = feature_value_df[DATE_COL].dt.strftime("%Y-%m-%d")
feature_value_df.to_csv(OUT_DIR / "stage2_final_selected_model_holdout_feature_values.csv", index=False)

shap_importance_df = (
    pd.DataFrame(
        {
            "feature_name": feature_cols,
            "mean_abs_shap": np.abs(shap_values).mean(axis=0),
            "mean_shap": shap_values.mean(axis=0),
            "feature_value_mean_holdout": holdout_df[feature_cols].mean().to_numpy(),
            "feature_value_std_holdout": holdout_df[feature_cols].std(ddof=0).to_numpy(),
        }
    )
    .sort_values("mean_abs_shap", ascending=False)
    .reset_index(drop=True)
)
shap_importance_df.to_csv(OUT_DIR / "stage2_final_selected_model_shap_importance.csv", index=False)

shap_importance_df


In [ ]:
summary_lines = [
    "# Main Rerun Stage 2 Final Model SHAP Summary",
    "",
    "## Final model",
    f"- `variant_key = {variant_key}`",
    f"- `model_name = {model_name}`",
    f"- `spec_name = {spec_name}`",
    f"- `best_params_json = {selected_holdout_row['best_params_json']}`",
    "",
    "## Sample",
    f"- `train_pool rows = {len(train_df)}`",
    f"- `primary_holdout rows = {len(holdout_df)}`",
    f"- `feature count = {len(feature_cols)}`",
    "",
    "## Holdout metrics",
    f"- `RMSE = {selected_holdout_row['rmse']:.6f}`",
    f"- `MAE = {selected_holdout_row['mae']:.6f}`",
    f"- `OOS R2 = {selected_holdout_row['oos_r2']:.6f}`",
    f"- `sign_accuracy = {selected_holdout_row['sign_accuracy']:.6f}`",
    "",
    "## Top Mean |SHAP| Features",
]

for _, row in shap_importance_df.head(10).iterrows():
    summary_lines.append(
        f"- `{row['feature_name']}`: `mean|SHAP| = {row['mean_abs_shap']:.6f}`, "
        f"`mean SHAP = {row['mean_shap']:.6f}`"
    )

summary_lines.extend(
    [
        "",
        "## Artifacts",
        f"- `summary_plot = {summary_plot_path.relative_to(PROJECT_ROOT)}`",
        f"- `shap_importance = {(OUT_DIR / 'stage2_final_selected_model_shap_importance.csv').relative_to(PROJECT_ROOT)}`",
        f"- `shap_values = {(OUT_DIR / 'stage2_final_selected_model_holdout_shap_values.csv').relative_to(PROJECT_ROOT)}`",
        f"- `feature_values = {(OUT_DIR / 'stage2_final_selected_model_holdout_feature_values.csv').relative_to(PROJECT_ROOT)}`",
        f"- `holdout_predictions = {(OUT_DIR / 'stage2_final_selected_model_holdout_predictions.csv').relative_to(PROJECT_ROOT)}`",
    ]
)

summary_path = OUT_DIR / "stage2_final_selected_model_shap_summary.md"
summary_path.write_text("\n".join(summary_lines) + "\n", encoding="utf-8")

print("Saved markdown summary to:", summary_path)
print()
print("\n".join(summary_lines))
